Assessing Risk of Measles Transmission By Airport through Total Measles Case Counts, Measures of Centrality, and MMR Vaccination Rates

In [53]:
import pandas as pd
import networkx as nx
from pathlib import Path
import numpy as np

# Go up one folder from code/ and into output/
output = Path.cwd().parent / "output"

edges = pd.read_csv(output / "airport_edges_2025.csv")
attrs = pd.read_csv(output / "airport_attributes_2025.csv", index_col="airport")

G = nx.from_pandas_edgelist(
    edges,
    "origin",
    "dest",
    edge_attr="passengers",
    create_using=nx.DiGraph
)

nx.set_node_attributes(G, attrs.to_dict("index"))

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 576
Edges: 8156


The Top 50 Airports by Measles Case Counts:

In [67]:
top_50_measles = attrs.sort_values(
    by="cases_2025",
    ascending=False
).head(50)

print(top_50_measles[["name", "state", "cases_2025"]])

                                                      name state  cases_2025
airport                                                                     
HOB                            Lea County Regional Airport    NM         501
GSP           Greenville-Spartanburg International Airport    SC         330
IFP                Laughlin Bullhead International Airport    AZ         216
SGU                             St George Regional Airport    UT         155
LBB            Lubbock Preston Smith International Airport    TX         144
ELP                          El Paso International Airport    TX          61
MAF               Midland International Air and Space Port    TX          52
GCK                           Garden City Regional Airport    KS          48
GRB                  Austin Straubel International Airport    WI          36
TYR                          Tyler Pounds Regional Airport    TX          29
PVU                                Provo Municipal Airport    UT          25

The Top 50 Airports by Betweeness Centrality:

In [68]:
bc = nx.betweenness_centrality(G)

top_50_bc = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:50]

bc_table = pd.DataFrame([
    {
        "Airport Code": node,
        "Airport": G.nodes[node]["name"],
        "State": G.nodes[node]["state"],
        "Betweenness Centrality": value
    }
    for node, value in top_50_bc
])

print(bc_table.to_string(index=False))

Airport Code                                                               Airport State  Betweenness Centrality
         ANC                           Ted Stevens Anchorage International Airport    AK                0.285674
         DEN                                          Denver International Airport    CO                0.147030
         ORD                                  Chicago O'Hare International Airport    IL                0.128274
         DFW                               Dallas Fort Worth International Airport    TX                0.121326
         BET                                                        Bethel Airport    AK                0.107837
         MSP Minneapolis–Saint Paul International Airport / Wold–Chamberlain Field    MN                0.058551
         ATL                      Hartsfield Jackson Atlanta International Airport    GA                0.056581
         SEA                                  Seattle–Tacoma International Airport    WA        

The Top 50 Airports by Eigenvector Centrality:

In [69]:
ec = nx.eigenvector_centrality(G)

top_50_ec = sorted(ec.items(), key=lambda x: x[1], reverse=True)[:50]

ec_table = pd.DataFrame([
    {
        "Airport Code": node,
        "Airport": G.nodes[node]["name"],
        "State": G.nodes[node]["state"],
        "Eigenvector Centrality": value
    }
    for node, value in top_50_ec
])

print(ec_table.to_string(index=False))

Airport Code                                                               Airport State  Eigenvector Centrality
         DFW                               Dallas Fort Worth International Airport    TX                0.169437
         ATL                      Hartsfield Jackson Atlanta International Airport    GA                0.167951
         DEN                                          Denver International Airport    CO                0.166967
         ORD                                  Chicago O'Hare International Airport    IL                0.166329
         CLT                               Charlotte Douglas International Airport    NC                0.154024
         LAS                                      Harry Reid International Airport    NV                0.152208
         MSP Minneapolis–Saint Paul International Airport / Wold–Chamberlain Field    MN                0.150450
         MCO                                         Orlando International Airport    FL        

The Top 50 Airports by Closeness Centrality:

In [70]:
cc = nx.closeness_centrality(G)

top_50_cc = sorted(cc.items(), key=lambda x: x[1], reverse=True)[:50]

cc_table = pd.DataFrame([
    {
        "Airport Code": node,
        "Airport": G.nodes[node]["name"],
        "State": G.nodes[node]["state"],
        "Closeness Centrality": value
    }
    for node, value in top_50_cc
])

print(cc_table.to_string(index=False))

Airport Code                                                               Airport State  Closeness Centrality
         DEN                                          Denver International Airport    CO              0.530708
         ORD                                  Chicago O'Hare International Airport    IL              0.528129
         DFW                               Dallas Fort Worth International Airport    TX              0.520043
         ATL                      Hartsfield Jackson Atlanta International Airport    GA              0.504592
         MSP Minneapolis–Saint Paul International Airport / Wold–Chamberlain Field    MN              0.500871
         LAS                                      Harry Reid International Airport    NV              0.499490
         PHX                              Phoenix Sky Harbor International Airport    AZ              0.489590
         SEA                                  Seattle–Tacoma International Airport    WA              0.487832
 

Airports that are both in the Top 50 Measles Case Counts as well as the Top 50 Betweeness, Eigenvector, or Closeness Centrality: 

In [71]:
measles_nodes = set(top_50_measles.index)

bc_nodes = {node for node, value in top_50_bc}
ec_nodes = {node for node, value in top_50_ec}
cc_nodes = {node for node, value in top_50_cc}

centrality_nodes = bc_nodes | ec_nodes | cc_nodes

common_nodes = measles_nodes & centrality_nodes

# Store centrality values
centrality_values = {
    "Betweenness": dict(top_50_bc),
    "Eigenvector": dict(top_50_ec),
    "Closeness": dict(top_50_cc)
}

results = []

for node in common_nodes:

    # Find which centrality measures include this airport
    measures = {
        measure: values[node]
        for measure, values in centrality_values.items()
        if node in values
    }

    # Select the measure with the highest centrality value
    highest_measure = max(measures, key=measures.get)
    highest_value = measures[highest_measure]

    results.append({
        "Airport Code": node,
        "Airport": G.nodes[node]["name"],
        "State": G.nodes[node]["state"],
        "Centrality Measure": highest_measure,
        "Centrality Value": highest_value,
        "Measles Cases": top_50_measles.loc[node, "cases_2025"]
    })

# Sort by highest centrality value
results = sorted(results, key=lambda x: x["Centrality Value"], reverse=True)

# Display as a DataFrame
common_airports_df = pd.DataFrame(results)

common_airports_df

,Airport Code,Airport,State,Centrality Measure,Centrality Value,Measles Cases
0,DEN,Denver International Airport,CO,Closeness,0.530708,13
1,DFW,Dallas Fort Worth International Airport,TX,Closeness,0.520043,6
2,ATL,Hartsfield Jackson Atlanta International Airport,GA,Closeness,0.504592,10
3,MSP,Minneapolis–Saint Paul International Airport /...,MN,Closeness,0.500871,14
4,IAH,George Bush Intercontinental Airport,TX,Closeness,0.479652,5
5,AUS,Austin Bergstrom International Airport,TX,Closeness,0.434409,11
6,MDW,Chicago Midway International Airport,IL,Closeness,0.430622,8
7,DAL,Dallas Love Field,TX,Closeness,0.426232,11
8,CMH,John Glenn Columbus International Airport,OH,Closeness,0.420298,24
9,HPN,Westchester County Airport,NY,Betweenness,0.011772,21


List of the Airports with Vaccination Rates Under 95%:

In [49]:
low_mmr_airports = (
    attrs
    .dropna(subset=["mmr_mean"])
    .query("mmr_mean <= 0.95")
    .sort_values("mmr_mean", ascending=True)
)

low_mmr_airports[["name", "state", "mmr_mean"]]

,name,state,mmr_mean
airport,,,
FAI,Fairbanks International Airport,AK,0.620000
LIH,Lihue Airport,HI,0.661600
SUN,Friedman Memorial Airport,ID,0.667500
JAC,Jackson Hole Airport,WY,0.670000
JHM,Kapalua Airport,HI,0.694300
...,...,...,...
CAE,Columbia Metropolitan Airport,SC,0.949000
SAF,Santa Fe Municipal Airport,NM,0.949040
MDW,Chicago Midway International Airport,IL,0.949500


Airports in the Top 50 Measles Counts, Top 50 Betweeness, Eigenvector, or Closeness Centrality, and with MMR Vaccination Rates Under 95%:

In [72]:
# Get airport codes from each group
measles_nodes = set(top_50_measles.index)

bc_nodes = {node for node, value in top_50_bc}
ec_nodes = {node for node, value in top_50_ec}
cc_nodes = {node for node, value in top_50_cc}

centrality_nodes = bc_nodes | ec_nodes | cc_nodes

# Airports with MMR vaccination rates <= 0.9
mmr_nodes = set(low_mmr_airports.index)

# Find airports appearing in all three groups
common_nodes = measles_nodes & centrality_nodes & mmr_nodes

# Store centrality values
centrality_values = {
    "Betweenness": dict(top_50_bc),
    "Eigenvector": dict(top_50_ec),
    "Closeness": dict(top_50_cc)
}

results = []

for node in common_nodes:

    # Find qualifying centrality values
    measures = {
        measure: values[node]
        for measure, values in centrality_values.items()
        if node in values
    }

    # Select highest centrality value
    highest_measure = max(measures, key=measures.get)
    highest_value = measures[highest_measure]

    results.append({
        "Airport Code": node,
        "Airport": G.nodes[node]["name"],
        "State": G.nodes[node]["state"],
        "Centrality Measure": highest_measure,
        "Centrality Value": highest_value,
        "Measles Cases": top_50_measles.loc[node, "cases_2025"],
        "MMR Vaccination Rate": attrs.loc[node, "mmr_mean"]
    })

# Sort by centrality value (highest to lowest)
results = sorted(
    results,
    key=lambda x: x["Centrality Value"],
    reverse=True
)

common_airports_df = pd.DataFrame(results)

common_airports_df

,Airport Code,Airport,State,Centrality Measure,Centrality Value,Measles Cases,MMR Vaccination Rate
0,DEN,Denver International Airport,CO,Closeness,0.530708,13,0.898850
1,DFW,Dallas Fort Worth International Airport,TX,Closeness,0.520043,6,0.919050
2,ATL,Hartsfield Jackson Atlanta International Airport,GA,Closeness,0.504592,10,0.914400
3,MSP,Minneapolis–Saint Paul International Airport /...,MN,Closeness,0.500871,14,0.861000
4,IAH,George Bush Intercontinental Airport,TX,Closeness,0.479652,5,0.939515
5,AUS,Austin Bergstrom International Airport,TX,Closeness,0.434409,11,0.933078
6,MDW,Chicago Midway International Airport,IL,Closeness,0.430622,8,0.949500
7,DAL,Dallas Love Field,TX,Closeness,0.426232,11,0.922426
8,CMH,John Glenn Columbus International Airport,OH,Closeness,0.420298,24,0.901200


In [51]:
# Convert DataFrame to LaTeX
latex_table = common_airports_df.to_latex(
    index=False,
    caption="Airports in the Top 100 Measles Cases and Centrality Rankings with MMR Vaccination Rates <= 0.9",
    label="tab:common_airports",
    float_format="%.4f",
    column_format="lllc rr".replace(" ", ""),
    escape=True
)

# Save as a .tex file
with open("common_airports.tex", "w") as f:
    f.write(latex_table)

print(latex_table)

\begin{table}
\caption{Airports in the Top 100 Measles Cases and Centrality Rankings with MMR Vaccination Rates <= 0.9}
\label{tab:common_airports}
\begin{tabular}{lllcrr}
\toprule
Airport Code & Airport & State & Centrality Measure & Centrality Value & Measles Cases & MMR Vaccination Rate \\
\midrule
DEN & Denver International Airport & CO & Closeness & 0.5307 & 13 & 0.8989 \\
DFW & Dallas Fort Worth International Airport & TX & Closeness & 0.5200 & 6 & 0.9191 \\
ATL & Hartsfield Jackson Atlanta International Airport & GA & Closeness & 0.5046 & 10 & 0.9144 \\
MSP & Minneapolis–Saint Paul International Airport / Wold–Chamberlain Field & MN & Closeness & 0.5009 & 14 & 0.8610 \\
IAH & George Bush Intercontinental Airport & TX & Closeness & 0.4797 & 5 & 0.9395 \\
AUS & Austin Bergstrom International Airport & TX & Closeness & 0.4344 & 11 & 0.9331 \\
MDW & Chicago Midway International Airport & IL & Closeness & 0.4306 & 8 & 0.9495 \\
DAL & Dallas Love Field & TX & Closeness & 0.4262 & 11 &